# ELEVATE SCORE V1 scoring engine

This notebook is the executable specification for the authoritative six-pillar ELEVATE SCORE V1 formula.

It keeps raw evidence separate from normalized scores, excludes unavailable metrics instead of treating them as zero, renormalizes weights at both metric and pillar levels, and reports data coverage/confidence separately from the score.

The score is a proprietary wellness and performance tracking tool. It is not intended to diagnose or treat medical conditions. Thresholds marked for professional validation must not be presented as clinically validated.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from statistics import mean
from typing import Any, Callable, Mapping, Sequence

ALGORITHM_VERSION = "elevate-v1.0.0"

PILLAR_WEIGHTS = {
    "recovery": 0.25,
    "body_composition": 0.20,
    "metabolic": 0.20,
    "cardio": 0.15,
    "strength": 0.10,
    "lifestyle": 0.10,
}

RECOVERY_WEIGHTS = {
    "sleep_duration": 0.20,
    "sleep_efficiency": 0.15,
    "hrv_trend": 0.20,
    "rhr_trend": 0.15,
    "readiness": 0.15,
    "stress": 0.10,
    "recovery_consistency": 0.05,
}
BODY_COMPOSITION_WEIGHTS = {
    "status": 0.40,
    "lean_mass_retention": 0.25,
    "visceral_fat": 0.20,
    "progress_trend": 0.15,
}
METABOLIC_WEIGHTS = {
    "waist_trend": 0.25,
    "energy_consistency": 0.20,
    "appetite": 0.20,
    "weight_rate": 0.20,
    "optional_data": 0.15,
}
CARDIO_WEIGHTS = {
    "vo2_max": 0.40,
    "rhr_trend": 0.20,
    "recovery_hr": 0.20,
    "zone_2_consistency": 0.20,
}
STRENGTH_WEIGHTS = {
    "strength_benchmarks": 0.35,
    "mobility": 0.25,
    "balance_core": 0.20,
    "training_consistency": 0.20,
}
LIFESTYLE_WEIGHTS = {
    "workout_attendance": 0.25,
    "recovery_attendance": 0.20,
    "nutrition_adherence": 0.25,
    "activity": 0.15,
    "routine_adherence": 0.15,
}


@dataclass(frozen=True)
class MetricEvidence:
    metric_name: str
    raw_value: float | int | None
    unit: str | None = None
    baseline_value: float | int | None = None
    baseline_period: str | None = None
    normalized_score: float | None = None
    data_source: str | None = None
    timestamp: str | None = None


def weighted_score(scores: Mapping[str, float | None], weights: Mapping[str, float]) -> float | None:
    available = [(scores.get(name), weight) for name, weight in weights.items() if scores.get(name) is not None and weight > 0]
    if not available:
        return None
    total_weight = sum(weight for _, weight in available)
    return sum(float(score) * weight for score, weight in available if score is not None) / total_weight


def calculate_data_coverage(scores: Mapping[str, float | None], weights: Mapping[str, float]) -> float:
    total_weight = sum(weight for weight in weights.values() if weight > 0)
    if total_weight == 0:
        return 0.0
    available_weight = sum(weight for name, weight in weights.items() if scores.get(name) is not None and weight > 0)
    return available_weight / total_weight


def confidence_label(coverage: float) -> str:
    if coverage >= 0.90:
        return "High"
    if coverage >= 0.75:
        return "Good"
    if coverage >= 0.50:
        return "Moderate"
    return "Limited"


def calculate_pillar(scores: Mapping[str, float | None], weights: Mapping[str, float]) -> dict[str, Any]:
    score = weighted_score(scores, weights)
    coverage = calculate_data_coverage(scores, weights)
    contributions = {
        name: {
            "score": scores.get(name),
            "weight": weight,
            "available": scores.get(name) is not None,
        }
        for name, weight in weights.items()
    }
    return {"score": score, "coverage": coverage, "confidence": confidence_label(coverage), "metrics": contributions}


def percent_change(current: float | None, baseline: float | None) -> float | None:
    if current is None or baseline is None or baseline <= 0:
        return None
    return ((current - baseline) / baseline) * 100


def score_sleep_duration(hours: float | None) -> float | None:
    if hours is None:
        return None
    if 7.5 <= hours <= 9.0:
        return 100
    if hours >= 7.0:
        return 85
    if hours >= 6.5:
        return 70
    if hours >= 6.0:
        return 55
    return 35


def score_sleep_efficiency(percent: float | None) -> float | None:
    if percent is None:
        return None
    if percent >= 90:
        return 100
    if percent >= 85:
        return 85
    if percent >= 80:
        return 70
    if percent >= 75:
        return 55
    return 35


def score_hrv_trend(change: float | None) -> float | None:
    if change is None:
        return None
    if change >= 10:
        return 100
    if change >= 5:
        return 90
    if change > -5:
        return 75
    if change >= -10:
        return 60
    return 40


def score_rhr_trend(change: float | None) -> float | None:
    if change is None:
        return None
    if change <= -5:
        return 100
    if change <= -2:
        return 90
    if change < 2:
        return 75
    if change <= 5:
        return 60
    return 40


def score_readiness(device_score: float | None) -> float | None:
    if device_score is None:
        return None
    if device_score >= 85:
        return 100
    if device_score >= 75:
        return 85
    if device_score >= 65:
        return 70
    if device_score >= 50:
        return 55
    return 35


def score_stress(level: int | None) -> float | None:
    return None if level is None else {1: 100, 2: 85, 3: 70, 4: 50, 5: 30}.get(level)


def score_recovery_consistency(days: int | None) -> float | None:
    if days is None:
        return None
    if days >= 6:
        return 100
    if days == 5:
        return 85
    if days == 4:
        return 70
    if days == 3:
        return 55
    return 35


def score_toward_target(current: float | None, target: float | None, direction: str | None = None) -> float | None:
    if current is None or target is None or target <= 0:
        return None
    distance = abs(current - target)
    if distance <= target * 0.02:
        return 100
    if distance <= target * 0.05:
        return 90
    if distance <= target * 0.10:
        return 75
    if distance <= target * 0.15:
        return 60
    return 40


def score_lean_mass_retention(
    baseline_lean_mass: float | None,
    current_lean_mass: float | None,
    baseline_weight: float | None,
    current_weight: float | None,
) -> float | None:
    if None in (baseline_lean_mass, current_lean_mass, baseline_weight, current_weight):
        return None
    assert baseline_lean_mass is not None and current_lean_mass is not None
    assert baseline_weight is not None and current_weight is not None
    weight_loss = baseline_weight - current_weight
    lean_mass_loss = baseline_lean_mass - current_lean_mass
    if weight_loss <= 0:
        return 75
    ratio = lean_mass_loss / weight_loss
    if ratio <= 0.10:
        return 100
    if ratio <= 0.20:
        return 90
    if ratio <= 0.30:
        return 75
    if ratio <= 0.40:
        return 60
    return 40


def score_visceral_fat(current: float | None, target: float | None) -> float | None:
    if current is None or target is None or target <= 0:
        return None
    ratio = current / target
    if ratio <= 1.00:
        return 100
    if ratio <= 1.10:
        return 90
    if ratio <= 1.20:
        return 75
    if ratio <= 1.35:
        return 60
    return 40


def score_improvement_trend(change: float | None, direction: str | None) -> float | None:
    if change is None:
        return None
    if direction == "lower_is_better":
        if change <= -10:
            return 100
        if change <= -5:
            return 90
        if change < 0:
            return 80
        if change <= 5:
            return 65
        return 45
    if direction == "higher_is_better":
        if change >= 10:
            return 100
        if change >= 5:
            return 90
        if change > 0:
            return 80
        if change >= -5:
            return 65
        return 45
    return None


def score_waist_trend(change: float | None) -> float | None:
    if change is None:
        return None
    if change <= -10:
        return 100
    if change <= -5:
        return 90
    if change < 0:
        return 80
    if change <= 5:
        return 65
    return 45


def score_energy_consistency(value: int | None) -> float | None:
    return None if value is None else {1: 40, 2: 55, 3: 70, 4: 85, 5: 100}.get(value)


def score_appetite(value: int | None) -> float | None:
    return None if value is None else {1: 40, 2: 55, 3: 70, 4: 85, 5: 100}.get(value)


def score_weight_rate(actual_rate: float | None, target_rate: float | None) -> float | None:
    if actual_rate is None or target_rate is None:
        return None
    difference = abs(actual_rate - target_rate)
    if difference <= 0.10:
        return 100
    if difference <= 0.25:
        return 90
    if difference <= 0.50:
        return 75
    if difference <= 0.75:
        return 60
    return 40


def score_optional_metabolic_data(score: float | None) -> float | None:
    return score


def score_vo2_max_percentile(percentile: float | None) -> float | None:
    if percentile is None:
        return None
    if percentile >= 90:
        return 100
    if percentile >= 75:
        return 90
    if percentile >= 50:
        return 75
    if percentile >= 25:
        return 60
    return 40


def score_recovery_hr(drop_bpm: float | None) -> float | None:
    if drop_bpm is None:
        return None
    if drop_bpm >= 30:
        return 100
    if drop_bpm >= 25:
        return 90
    if drop_bpm >= 20:
        return 75
    if drop_bpm >= 12:
        return 60
    return 40


def score_zone_2_days(days_per_week: int | None) -> float | None:
    if days_per_week is None:
        return None
    if days_per_week >= 4:
        return 100
    if days_per_week == 3:
        return 90
    if days_per_week == 2:
        return 75
    if days_per_week == 1:
        return 60
    return 40


def score_strength_benchmark(percentile: float | None) -> float | None:
    return score_vo2_max_percentile(percentile)


def average_available(scores: Sequence[float | None] | None) -> float | None:
    if scores is None:
        return None
    available = [score for score in scores if score is not None]
    return mean(available) if available else None


def score_training_consistency(adherence_percent: float | None) -> float | None:
    if adherence_percent is None:
        return None
    if adherence_percent >= 90:
        return 100
    if adherence_percent >= 80:
        return 90
    if adherence_percent >= 70:
        return 75
    if adherence_percent >= 60:
        return 60
    return 40


def score_attendance(actual: int | None, scheduled: int | None) -> float | None:
    if actual is None or scheduled is None or scheduled <= 0:
        return None
    return score_training_consistency((actual / scheduled) * 100)


def score_nutrition_adherence(percent: float | None) -> float | None:
    return score_training_consistency(percent)


def score_activity(actual_average: float | None, target_average: float | None) -> float | None:
    if actual_average is None or target_average is None or target_average <= 0:
        return None
    ratio = actual_average / target_average
    if ratio >= 1.00:
        return 100
    if ratio >= 0.90:
        return 90
    if ratio >= 0.80:
        return 75
    if ratio >= 0.70:
        return 60
    return 40


def score_routine_adherence(percent: float | None) -> float | None:
    return score_training_consistency(percent)


print(f"Loaded {ALGORITHM_VERSION}: {len(PILLAR_WEIGHTS)} pillars")

Loaded elevate-v1.0.0: 6 pillars


## Pillar and total calculations

Each pillar returns its unrounded score, metric-level coverage, confidence label, and every weighted contribution. The overall calculation excludes missing pillars, renormalizes the remaining pillar weights, and computes overall data confidence from the original possible weights.

In [2]:
def calculate_recovery(data: Mapping[str, Any]) -> dict[str, Any]:
    hrv_change = percent_change(data.get("hrv_7d_median"), data.get("hrv_28d_median"))
    rhr_change = percent_change(data.get("rhr_7d_value"), data.get("rhr_28d_baseline"))
    scores = {
        "sleep_duration": score_sleep_duration(data.get("sleep_hours")),
        "sleep_efficiency": score_sleep_efficiency(data.get("sleep_efficiency")),
        "hrv_trend": score_hrv_trend(hrv_change),
        "rhr_trend": score_rhr_trend(rhr_change),
        "readiness": score_readiness(data.get("normalized_readiness")),
        "stress": score_stress(data.get("stress_level")),
        "recovery_consistency": score_recovery_consistency(data.get("recovery_days")),
    }
    result = calculate_pillar(scores, RECOVERY_WEIGHTS)
    result["derived"] = {"hrv_percent_change": hrv_change, "rhr_percent_change": rhr_change}
    return result


def calculate_body_composition(data: Mapping[str, Any]) -> dict[str, Any]:
    scores = {
        "status": score_toward_target(data.get("current_status"), data.get("target_status"), data.get("status_direction")),
        "lean_mass_retention": score_lean_mass_retention(
            data.get("baseline_lean_mass"),
            data.get("current_lean_mass"),
            data.get("baseline_weight"),
            data.get("current_weight"),
        ),
        "visceral_fat": score_visceral_fat(data.get("visceral_fat"), data.get("visceral_fat_target")),
        "progress_trend": score_improvement_trend(data.get("progress_percent_change"), data.get("progress_direction")),
    }
    return calculate_pillar(scores, BODY_COMPOSITION_WEIGHTS)


def calculate_metabolic(data: Mapping[str, Any]) -> dict[str, Any]:
    scores = {
        "waist_trend": score_waist_trend(data.get("waist_percent_change")),
        "energy_consistency": score_energy_consistency(data.get("energy_consistency")),
        "appetite": score_appetite(data.get("appetite_control")),
        "weight_rate": score_weight_rate(data.get("actual_weight_rate"), data.get("target_weight_rate")),
        "optional_data": score_optional_metabolic_data(data.get("optional_metabolic_score")),
    }
    return calculate_pillar(scores, METABOLIC_WEIGHTS)


def calculate_cardio(data: Mapping[str, Any], rhr_trend_score: float | None) -> dict[str, Any]:
    scores = {
        "vo2_max": score_vo2_max_percentile(data.get("vo2_percentile")),
        "rhr_trend": rhr_trend_score,
        "recovery_hr": score_recovery_hr(data.get("recovery_hr_drop_bpm")),
        "zone_2_consistency": score_zone_2_days(data.get("zone_2_days")),
    }
    return calculate_pillar(scores, CARDIO_WEIGHTS)


def calculate_strength(data: Mapping[str, Any]) -> dict[str, Any]:
    scores = {
        "strength_benchmarks": score_strength_benchmark(data.get("strength_percentile")),
        "mobility": average_available(data.get("mobility_test_scores")),
        "balance_core": average_available(data.get("balance_core_test_scores")),
        "training_consistency": score_training_consistency(data.get("training_adherence_percent")),
    }
    return calculate_pillar(scores, STRENGTH_WEIGHTS)


def calculate_lifestyle(data: Mapping[str, Any]) -> dict[str, Any]:
    scores = {
        "workout_attendance": score_attendance(data.get("workouts_actual"), data.get("workouts_scheduled")),
        "recovery_attendance": score_attendance(data.get("recovery_actual"), data.get("recovery_scheduled")),
        "nutrition_adherence": score_nutrition_adherence(data.get("nutrition_adherence_percent")),
        "activity": score_activity(data.get("activity_average"), data.get("activity_target")),
        "routine_adherence": score_routine_adherence(data.get("routine_adherence_percent")),
    }
    return calculate_pillar(scores, LIFESTYLE_WEIGHTS)


def calculate_momentum(daily_total_scores: Sequence[float | None]) -> float | None:
    """Current 28-day mean minus prior 28-day mean; requires 56 complete daily scores."""
    if len(daily_total_scores) < 56:
        return None
    prior = daily_total_scores[-56:-28]
    current = daily_total_scores[-28:]
    if any(score is None for score in prior) or any(score is None for score in current):
        return None
    return round(mean(float(score) for score in current if score is not None) - mean(float(score) for score in prior if score is not None), 1)


def calculate_elevate_score(
    pillars: Mapping[str, Mapping[str, Any]],
    evidence: Sequence[MetricEvidence] = (),
    daily_total_scores: Sequence[float | None] = (),
) -> dict[str, Any]:
    pillar_scores = {
        name: pillars.get(name, {}).get("score")
        for name in PILLAR_WEIGHTS
    }
    raw_score = weighted_score(pillar_scores, PILLAR_WEIGHTS)
    data_confidence = sum(
        PILLAR_WEIGHTS[name] * float(pillars.get(name, {}).get("coverage", 0.0))
        for name in PILLAR_WEIGHTS
    ) / sum(PILLAR_WEIGHTS.values())
    return {
        "algorithm_version": ALGORITHM_VERSION,
        "score": None if raw_score is None else round(raw_score, 1),
        "raw_score": raw_score,
        "momentum": calculate_momentum(daily_total_scores),
        "data_confidence": round(data_confidence, 4),
        "confidence_label": confidence_label(data_confidence),
        "pillars": dict(pillars),
        "evidence": [asdict(item) for item in evidence],
    }


def calculate_all_pillars(inputs: Mapping[str, Mapping[str, Any]]) -> dict[str, dict[str, Any]]:
    recovery = calculate_recovery(inputs.get("recovery", {}))
    rhr_score = recovery["metrics"]["rhr_trend"]["score"]
    return {
        "recovery": recovery,
        "body_composition": calculate_body_composition(inputs.get("body_composition", {})),
        "metabolic": calculate_metabolic(inputs.get("metabolic", {})),
        "cardio": calculate_cardio(inputs.get("cardio", {}), rhr_score),
        "strength": calculate_strength(inputs.get("strength", {})),
        "lifestyle": calculate_lifestyle(inputs.get("lifestyle", {})),
    }


print("Pillar calculators ready")

Pillar calculators ready


## Deterministic formula checks

These checks lock the V1 threshold boundaries and both levels of missing-data behavior. **Resolved:** the source document's worked recovery result of 81.5 is an arithmetic error. Its stated values and weights produce a numerator of 52.25 and an available weight of 0.65, so the authoritative V1 result is `52.25 / 0.65 = 80.3846`, displayed as **80.4**.

In [4]:
def score_weight_rate(actual_rate: float | None, target_rate: float | None) -> float | None:
    if actual_rate is None or target_rate is None:
        return None
    # Decimal inputs such as 1.1 - 1.0 can exceed 0.10 by a binary floating-point epsilon.
    difference = round(abs(actual_rate - target_rate), 10)
    if difference <= 0.10:
        return 100
    if difference <= 0.25:
        return 90
    if difference <= 0.50:
        return 75
    if difference <= 0.75:
        return 60
    return 40


print("Weight-rate boundaries normalized")

Weight-rate boundaries normalized


In [8]:
# Recovery and trend boundaries
assert score_sleep_duration(None) is None
assert [score_sleep_duration(value) for value in (5.9, 6.0, 6.5, 7.0, 7.5, 9.0, 9.1)] == [35, 55, 70, 85, 100, 100, 85]
assert [score_sleep_efficiency(value) for value in (74.9, 75, 80, 85, 90)] == [35, 55, 70, 85, 100]
assert [score_hrv_trend(value) for value in (-10.1, -10, -5, -4.9, 5, 10)] == [40, 60, 60, 75, 90, 100]
assert [score_rhr_trend(value) for value in (-5, -2, 1.9, 2, 5, 5.1)] == [100, 90, 75, 60, 60, 40]
assert [score_readiness(value) for value in (49, 50, 65, 75, 85)] == [35, 55, 70, 85, 100]
assert [score_stress(value) for value in (1, 2, 3, 4, 5, 6)] == [100, 85, 70, 50, 30, None]
assert [score_recovery_consistency(value) for value in (2, 3, 4, 5, 6)] == [35, 55, 70, 85, 100]

# Body composition and metabolic boundaries
assert [score_toward_target(value, 100) for value in (102, 105, 110, 115, 116)] == [100, 90, 75, 60, 40]
assert score_lean_mass_retention(60, 59.5, 80, 75) == 100
assert score_lean_mass_retention(60, 58, 80, 75) == 60
assert [score_visceral_fat(value, 10) for value in (10, 11, 12, 13.5, 14)] == [100, 90, 75, 60, 40]
assert [score_improvement_trend(value, "lower_is_better") for value in (-10, -5, -0.1, 5, 5.1)] == [100, 90, 80, 65, 45]
assert [score_improvement_trend(value, "higher_is_better") for value in (-5.1, -5, 0.1, 5, 10)] == [45, 65, 80, 90, 100]
assert [score_waist_trend(value) for value in (-10, -5, -0.1, 5, 5.1)] == [100, 90, 80, 65, 45]
assert [score_energy_consistency(value) for value in (1, 2, 3, 4, 5)] == [40, 55, 70, 85, 100]
assert [score_appetite(value) for value in (1, 2, 3, 4, 5)] == [40, 55, 70, 85, 100]
assert [score_weight_rate(value, 1.0) for value in (1.1, 1.25, 1.5, 1.75, 1.76)] == [100, 90, 75, 60, 40]

# Cardio, strength, and lifestyle boundaries
assert [score_vo2_max_percentile(value) for value in (24, 25, 50, 75, 90)] == [40, 60, 75, 90, 100]
assert [score_recovery_hr(value) for value in (11, 12, 20, 25, 30)] == [40, 60, 75, 90, 100]
assert [score_zone_2_days(value) for value in (0, 1, 2, 3, 4)] == [40, 60, 75, 90, 100]
assert [score_training_consistency(value) for value in (59, 60, 70, 80, 90)] == [40, 60, 75, 90, 100]
assert score_attendance(8, 10) == 90
assert [score_activity(value, 10000) for value in (6999, 7000, 8000, 9000, 10000)] == [40, 60, 75, 90, 100]
assert average_available([80, None, 100]) == 90

# Metric-level renormalization: the source document labels this 81.5, but the formula yields 80.4.
recovery_example = weighted_score(
    {
        "sleep_duration": 85,
        "sleep_efficiency": 85,
        "hrv_trend": None,
        "rhr_trend": 75,
        "readiness": None,
        "stress": 70,
        "recovery_consistency": 85,
    },
    RECOVERY_WEIGHTS,
)
assert recovery_example is not None and round(recovery_example, 1) == 80.4
assert calculate_data_coverage(
    {"sleep_duration": 85, "sleep_efficiency": 85, "rhr_trend": 75, "stress": 70, "recovery_consistency": 85},
    RECOVERY_WEIGHTS,
) == 0.65

body_example = weighted_score(
    {"status": 75, "lean_mass_retention": 90, "visceral_fat": None, "progress_trend": 85},
    BODY_COMPOSITION_WEIGHTS,
)
assert body_example is not None and round(body_example, 1) == 81.6

# Pillar-level renormalization with metabolic entirely unavailable.
def test_pillar(score: float | None, coverage: float = 1.0) -> dict[str, Any]:
    return {"score": score, "coverage": coverage, "metrics": {}}

missing_pillar_result = calculate_elevate_score({
    "recovery": test_pillar(82),
    "body_composition": test_pillar(74),
    "metabolic": test_pillar(None, 0.0),
    "cardio": test_pillar(78),
    "strength": test_pillar(70),
    "lifestyle": test_pillar(88),
})
assert missing_pillar_result["score"] == 78.5
assert missing_pillar_result["data_confidence"] == 0.8

# Confidence, no-data, rounding, evidence, and momentum behavior.
assert [confidence_label(value) for value in (0.49, 0.50, 0.75, 0.90)] == ["Limited", "Moderate", "Good", "High"]
assert weighted_score({"x": None}, {"x": 1.0}) is None
assert calculate_data_coverage({"x": None}, {"x": 1.0}) == 0.0
assert calculate_elevate_score({})["score"] is None
assert calculate_momentum([70.0] * 28 + [74.8] * 28) == 4.8
assert calculate_momentum([70.0] * 55) is None
assert calculate_elevate_score({"recovery": test_pillar(77.34)})["score"] == 77.3

print("All ELEVATE V1 formula checks passed")
print("Documented recovery example result:", round(recovery_example, 1))
print("Documented body composition example result:", round(body_example, 1))
print("Documented missing-pillar result:", missing_pillar_result["score"])

All ELEVATE V1 formula checks passed
Documented recovery example result: 80.4
Documented body composition example result: 81.6
Documented missing-pillar result: 78.5


## Open Wearables input contract

**Available from the audited normalized API, with source selection and local-day handling still required:**

- Sleep duration and efficiency from sleep sessions
- Daily HRV and resting heart rate for 7-day versus prior 28-day medians; SDNN and RMSSD must never be mixed
- Provider recovery/readiness scores, after a provider-specific normalization contract is approved
- Weight, body-fat percentage, VO2 max estimates, steps, workouts, and heart-rate samples

**Supported by the platform but absent or unproven in the audited seed:** lean body mass, skeletal muscle mass, waist circumference, hydration, and one-minute heart-rate recovery.

**Must come from Elevate configuration, questionnaires, or standardized assessments:** stress, energy, appetite, nutrition, routines, attendance schedules, client targets, age/sex VO2 reference percentiles, visceral-fat targets, recovery test protocol, Zone 2 classification, strength benchmarks, mobility, balance/core tests, and goal-specific weight-rate targets.

An unavailable or unapproved input must be `None`. It is excluded from the score and lowers data confidence; it is never guessed or converted to zero.

## Synthetic end-to-end example

This example exists only to exercise all six pillars. Its manual assessments, targets, percentiles, and provider values are fabricated and must not be used as defaults.

In [9]:
synthetic_inputs = {
    "recovery": {
        "sleep_hours": 7.2,
        "sleep_efficiency": 88,
        "hrv_7d_median": 42,
        "hrv_28d_median": 40,
        "rhr_7d_value": 58,
        "rhr_28d_baseline": 60,
        "normalized_readiness": 79,
        "stress_level": 3,
        "recovery_days": 5,
    },
    "body_composition": {
        "current_status": 24,
        "target_status": 22,
        "status_direction": "lower_is_better",
        "baseline_lean_mass": 62,
        "current_lean_mass": 61.5,
        "baseline_weight": 85,
        "current_weight": 80,
        "visceral_fat": None,
        "visceral_fat_target": None,
        "progress_percent_change": -5,
        "progress_direction": "lower_is_better",
    },
    "metabolic": {
        "waist_percent_change": -4,
        "energy_consistency": 4,
        "appetite_control": 4,
        "actual_weight_rate": -0.7,
        "target_weight_rate": -0.6,
        "optional_metabolic_score": None,
    },
    "cardio": {
        "vo2_percentile": 72,
        "recovery_hr_drop_bpm": None,
        "zone_2_days": 3,
    },
    "strength": {
        "strength_percentile": 68,
        "mobility_test_scores": [75, 80, None],
        "balance_core_test_scores": [85, 75],
        "training_adherence_percent": 84,
    },
    "lifestyle": {
        "workouts_actual": 9,
        "workouts_scheduled": 10,
        "recovery_actual": 4,
        "recovery_scheduled": 5,
        "nutrition_adherence_percent": 82,
        "activity_average": 8200,
        "activity_target": 9000,
        "routine_adherence_percent": 76,
    },
}

synthetic_evidence = [
    MetricEvidence(
        metric_name="hrv",
        raw_value=42,
        unit="ms",
        baseline_value=40,
        baseline_period="28-day median",
        normalized_score=90,
        data_source="synthetic_oura",
        timestamp="2026-09-15T08:00:00Z",
    ),
    MetricEvidence(
        metric_name="sleep_duration",
        raw_value=7.2,
        unit="hours",
        normalized_score=85,
        data_source="synthetic_oura",
        timestamp="2026-09-15T08:00:00Z",
    ),
]

synthetic_pillars = calculate_all_pillars(synthetic_inputs)
synthetic_result = calculate_elevate_score(
    synthetic_pillars,
    evidence=synthetic_evidence,
    daily_total_scores=[70.0] * 28 + [74.8] * 28,
)

print("ELEVATE SCORE:", synthetic_result["score"])
print("MOMENTUM:", synthetic_result["momentum"])
print("DATA CONFIDENCE:", f"{synthetic_result['data_confidence']:.0%}", synthetic_result["confidence_label"])
for name, pillar in synthetic_result["pillars"].items():
    display_score = None if pillar["score"] is None else round(pillar["score"], 1)
    print(f"{name:20} score={display_score!s:>5} coverage={pillar['coverage']:.0%}")

ELEVATE SCORE: 85.2
MOMENTUM: 4.8
DATA CONFIDENCE: 90% High
recovery             score= 85.2 coverage=100%
body_composition     score= 85.6 coverage=80%
metabolic            score= 87.1 coverage=85%
cardio               score= 82.5 coverage=80%
strength             score= 79.6 coverage=100%
lifestyle            score= 90.2 coverage=100%


## Prototype V1 rules finalized

These rules are locked for synthetic testing. They are deterministic product rules, not validated clinical standards.

1. **VO2 max reference:** the scoring engine accepts an age/sex-adjusted percentile from 0 to 100, then applies the V1 percentile bands. Synthetic fixtures may provide the percentile directly. A raw VO2 max value without an approved reference table, age, and sex produces `None`.
2. **Body-composition target:** use an explicit client target approved outside the scoring engine. Current and target values must use the same metric, unit, device/method, and measurement conditions. No universal body-fat target is inferred. Synthetic fixtures must declare their target.
3. **Heart-rate recovery protocol:** use the BPM drop from heart rate at exercise cessation to heart rate exactly 60 seconds later during passive seated recovery after a documented standardized test. Values from other intervals, active cooldowns, or unknown protocols produce `None`. Repeated comparisons must use the same protocol.
4. **Strength tests:** the engine accepts one 0-100 composite percentile supplied by an approved assessment battery. Synthetic fixtures may provide this percentile directly. Raw lift values are not converted until the Lead PT defines exercises, technique, scaling, and reference populations.
5. **Weight-rate goal:** both actual and target rates use signed percentage of body weight per week: loss is negative, maintenance is zero, and gain is positive. Actual rate is calculated from a 28-day weight trend; the client-specific target must be explicit. No default loss or gain target is inferred.
6. **Provider readiness normalization:** Oura Readiness, WHOOP Recovery, and Ultrahuman Recovery values are accepted only when documented on a 0-100 scale and are identity-mapped to `normalized_readiness`. Use one configured primary provider per client; never average providers. Reject values outside 0-100, unknown score categories, and readings older than 48 hours.

Still required before production: approve the actual VO2 reference dataset and licensing, body-composition targets, strength battery and norms, standardized exercise test, client weight-rate targets, and provider contracts against real payloads.

Resolved formula note: the worked recovery example's **81.5** was an arithmetic error. ELEVATE V1 uses the stated weighted formula and therefore returns **80.4** for those inputs.